# 11. Idiomas por persona

Proyecto de tesis (Maestria en Ciencia de Datos e IA, ESPOL): *Sistema de Generacion de Perfiles del Personal Docente y Administrativo en ESPOL para la asignacion inteligente de tareas*. Este notebook corresponde a la **Fase 1 y 2** de la metodologia: extraccion y depuracion de una de las fuentes institucionales que alimentan el catalogo de variables (historia laboral, capacitaciones, proyectos, experiencia y direcciones de tesis) usado para construir los perfiles multidimensionales del personal.

**Fuente:** `data/raw/idiomaspersonas.csv`  
**Salida:** `data/processed/idiomas_personas.csv`

Nivel de idiomas (lectura, escritura, comprension, conversacion) declarado por cada persona. El archivo de origen duplica IDPERSONA/IDIDIOMA/ULTIMO_CAMBIO/VERSION por un join previo; se colapsan antes de limpiar.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [2]:
df = pc.leer_csv('idiomaspersonas.csv', low_memory=False)
df.head()

Leido idiomaspersonas.csv con encoding=utf-8-sig -> 4137 filas, 20 columnas


,IDPERSONA,IDIDIOMA,IDIOMA,ULTIMO_CAMBIO,VERSION,IDIDIOMAPERSONA,IDPERSONA.1,IDIDIOMA.1,NIVELLECTURA,NIVELESCRITURA,NIVELCOMPRESION,NIVELCONVERSACION,ULTIMO_CAMBIO.1,VERSION.1,NAMEARCHDOC,DESCRIPCION,REFARCHIVO,LENGUANATIVA,NIVELMCER,FECHASUBIDAARCHIVO
0,26,2,INGLES,2006-05-30-12.44.15.284836,1,491,26,2,I,I,NaN,I,2015-11-20-18.57.49.498542,1,IDI0102921053-2.PDF,NaN,"561,525",NaN,NaN,NaN
1,38,6,FRANCES,2006-06-07-15.27.51.941263,1,3125,38,6,I,I,B,A,2021-04-22-14.35.40.084590,1,NaN,NaN,NaN,S,NaN,NaN
2,38,2,INGLES,2006-05-30-12.44.15.284836,1,3127,38,2,I,I,B,I,2021-04-22-14.36.02.893052,1,NaN,NaN,NaN,S,NaN,NaN
3,86,3,PORTUGUES,2006-05-30-12.46.53.950442,1,777,86,3,A,A,NaN,A,2016-05-25-11.28.08.510614,1,NaN,NaN,"44703,43837",NaN,NaN,NaN
4,94,2,INGLES,2006-05-30-12.44.15.284836,1,352,94,2,A,A,NaN,I,2016-10-20-11.32.48.470390,1,IDI0602157588-2.PDF,NaN,"463,438",N,NaN,NaN


## 2. Exploración inicial

In [3]:
pc.resumen(df, 'idiomas_personas')

--- Resumen idiomas_personas ---
Dimensiones: 4137 filas x 20 columnas
Filas duplicadas: 0
Columnas con nulos (%):
DESCRIPCION           99.2
FECHASUBIDAARCHIVO    94.1
NAMEARCHDOC           91.6
NIVELMCER             84.6
REFARCHIVO            72.4
NIVELCOMPRESION       46.7
LENGUANATIVA           8.0
dtype: float64


In [4]:
df.dtypes

IDPERSONA             int64
IDIDIOMA              int64
IDIOMA                  str
ULTIMO_CAMBIO           str
VERSION               int64
IDIDIOMAPERSONA       int64
IDPERSONA.1           int64
IDIDIOMA.1            int64
NIVELLECTURA            str
NIVELESCRITURA          str
NIVELCOMPRESION         str
NIVELCONVERSACION       str
ULTIMO_CAMBIO.1         str
VERSION.1             int64
NAMEARCHDOC             str
DESCRIPCION             str
REFARCHIVO              str
LENGUANATIVA            str
NIVELMCER               str
FECHASUBIDAARCHIVO      str
dtype: object

## 3. Limpieza

In [5]:
df = pc.limpiar_strings(df)
df = pc.quitar_columnas_duplicadas(df)
df = pc.quitar_columnas_vacias(df, umbral=0.99)
df = pc.quitar_columnas_constantes(df)

Columnas duplicadas colapsadas (se conserva la primera aparicion): ['IDPERSONA', 'IDIDIOMA', 'ULTIMO_CAMBIO', 'VERSION']
Columnas eliminadas por tener >= 99% de nulos: ['DESCRIPCION']
Columnas eliminadas por ser constantes: ['VERSION', 'NIVELCOMPRESION']


## 5. Tipado de fechas e identificadores

In [6]:
df = pc.castear_fechas(df, ['ULTIMO_CAMBIO', 'FECHASUBIDAARCHIVO'])
df = pc.castear_enteros(df, ['IDPERSONA', 'IDIDIOMA', 'IDIDIOMAPERSONA'])

## 7. Verificación final

In [7]:
pc.resumen(df, 'idiomas_personas (procesado)')
df.head()

--- Resumen idiomas_personas (procesado) ---
Dimensiones: 4137 filas x 13 columnas
Filas duplicadas: 0
Columnas con nulos (%):
FECHASUBIDAARCHIVO    94.1
NAMEARCHDOC           91.6
NIVELMCER             84.6
REFARCHIVO            72.4
LENGUANATIVA           8.0
dtype: float64


,IDPERSONA,IDIDIOMA,IDIOMA,ULTIMO_CAMBIO,IDIDIOMAPERSONA,NIVELLECTURA,NIVELESCRITURA,NIVELCONVERSACION,NAMEARCHDOC,REFARCHIVO,LENGUANATIVA,NIVELMCER,FECHASUBIDAARCHIVO
0,26,2,INGLES,2006-05-30 12:44:15.284836,491,I,I,I,IDI0102921053-2.PDF,"561,525",<NA>,<NA>,NaT
1,38,6,FRANCES,2006-06-07 15:27:51.941263,3125,I,I,A,<NA>,<NA>,S,<NA>,NaT
2,38,2,INGLES,2006-05-30 12:44:15.284836,3127,I,I,I,<NA>,<NA>,S,<NA>,NaT
3,86,3,PORTUGUES,2006-05-30 12:46:53.950442,777,A,A,A,<NA>,"44703,43837",<NA>,<NA>,NaT
4,94,2,INGLES,2006-05-30 12:44:15.284836,352,A,A,I,IDI0602157588-2.PDF,"463,438",N,<NA>,NaT


## 8. Guardado en data/processed

In [8]:
pc.guardar_procesado(df, 'idiomas_personas.csv')

Guardado: D:\Proyecto_Tesis\data\processed\idiomas_personas.csv (4137 filas x 13 columnas)


WindowsPath('D:/Proyecto_Tesis/data/processed/idiomas_personas.csv')